## Plot TOP 50 TLD Distributions of DomainList


In [1]:
from pathlib import Path

import altair as alt
import polars as pl

DATA_DIR = Path("../../data/measurements")

df = (
    pl.read_parquet(DATA_DIR / "domain_sourcing.parquet")
    .with_columns(pl.col("domain").str.split(".").list.last().str.split(":").list.first().alias("tld"))
    .group_by("tld")
    .agg(pl.count("domain").cast(pl.Float64).alias("count"))
    .sort("count", descending=True)
)

In [2]:
# Create the Altair chart with log10 y-axis, no grid, no axis lines, no border, and modified x-axis title
chart = (
    alt.Chart(df.slice(0, 25))
    .mark_bar()
    .encode(
        x=alt.X("tld:N", title="Top-Level Domain", sort="-y"),
        y=alt.Y("count:Q", title="Domains", axis=alt.Axis(grid=False)),
        tooltip=["tld", "count"],
    )
    .properties(title="Top-Level Domain Distribution - Top 25 TLDs")
)

# Display the chart
chart.show()

alt.Chart(...)

In [3]:
# Create the Altair chart with log10 y-axis, no grid, no axis lines, no border, and modified x-axis title
chart = (
    alt.Chart(
        df.with_columns(((pl.col("count") / df.get_column("count").sum()) * 100).alias("percentage"))
        .sort("count", descending=True)
        .slice(0, 25)
    )
    .mark_bar()
    .encode(
        x=alt.X("tld:N", title="Top-Level Domain", sort="-y"),
        y=alt.Y("percentage:Q", title="Percentage", axis=alt.Axis(grid=False), scale=alt.Scale(domain=[0, 100])),
        tooltip=["tld", "percentage"],
    )
    .properties(title="Top-Level Domain Distribution - Top 25 TLDs")
)

# Display the chart
chart.show()

alt.Chart(...)